# Auditoria de imagenes nuevas contra mosaic dataset

Notebook de control para comparar una carpeta raiz de entrada contra un mosaic dataset, usando una subcarpeta de confianza indicada por el cliente.


In [19]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_image_audit as mosaic_audit
mosaic_audit = importlib.reload(mosaic_audit)
from core.mosaic_image_audit import *

# PARAMETROS PRINCIPALES
PATH_INPUT_RAIZ = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT"
PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

# Subfolder de confianza informado por el cliente dentro de PATH_INPUT_RAIZ.
SUBFOLDER_CONTROL_CLIENTE = "20260206_Geosupport_primera entrega"

# Usar None para leer toda la tabla exportada. Puede bajarse para pruebas rapidas.
MAX_EXPORTED_PATH_ROWS = None

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_results_dir = Path.cwd() / "outputs" / "auditoria_mosaico" / run_timestamp

print("Modulo auditoria:", mosaic_audit.__file__)
print("Version logica:", AUDIT_LOGIC_VERSION)
print("Input raiz:", PATH_INPUT_RAIZ)
print("Subfolder control cliente:", SUBFOLDER_CONTROL_CLIENTE)
print("Mosaic dataset:", PATH_MOSAIC_DATASET)
print("Salida:", output_results_dir)

Modulo auditoria: c:\Users\esrlrivero_adm\Documents\amsa-pao-geosupport\core\mosaic_image_audit.py
Version logica: 2026-06-12-parser-fix-export-mosaic-paths
Input raiz: \\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT
Subfolder control cliente: 20260206_Geosupport_primera entrega
Mosaic dataset: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport
Salida: c:\Users\esrlrivero_adm\Documents\amsa-pao-geosupport\outputs\auditoria_mosaico\20260612_162412


## 1. Buscar imagenes en el input raiz

La busqueda es recursiva. El flag `is_in_control_folder` identifica las imagenes dentro de la entrega que el cliente marco como confiable.


In [20]:
input_images_df = scan_input_images(PATH_INPUT_RAIZ)
input_images_df = add_control_flags(input_images_df, SUBFOLDER_CONTROL_CLIENTE)

ortho_input_images_df = input_images_df[input_images_df["extension"].isin(ORTHO_MOSAIC_EXTENSIONS)].copy()
control_ortho_images_df = ortho_input_images_df[ortho_input_images_df["is_in_control_folder"]].copy()

print(f"Imagenes encontradas en input raiz: {len(input_images_df)}")
print(f"TIF/TIFF en input raiz: {len(ortho_input_images_df)}")
print(f"TIF/TIFF en subfolder de control cliente: {len(control_ortho_images_df)}")

display(input_images_df.groupby(["extension", "is_in_control_folder"]).size().reset_index(name="count"))
display(ortho_input_images_df.groupby(["top_folder", "is_in_control_folder"]).size().reset_index(name="count"))

Imagenes encontradas en input raiz: 1999
TIF/TIFF en input raiz: 66
TIF/TIFF en subfolder de control cliente: 33


,extension,is_in_control_folder,count
0,.jpg,False,1359
1,.jpg,True,574
2,.tif,False,33
3,.tif,True,33


,top_folder,is_in_control_folder,count
0,20260206_Geosupport_primera entrega,True,33
1,20260519_Geosupport,False,33


## 2. Exportar paths oficiales del mosaic dataset

Se usa `arcpy.management.ExportMosaicDatasetPaths` para obtener los nombres/rutas reales que quedaron cargados en el mosaico. Esta tabla es la fuente de control para comparar contra el nombre esperado que genera el script.


In [21]:
mosaic_paths_df, mosaic_paths_fields_df, exported_mosaic_paths_table = export_mosaic_dataset_paths_to_dataframe(
    PATH_MOSAIC_DATASET,
    max_rows=MAX_EXPORTED_PATH_ROWS,
)
candidate_path_fields = detect_candidate_path_fields(mosaic_paths_fields_df)
mosaic_image_inventory_df = build_mosaic_image_inventory(mosaic_paths_df, candidate_path_fields)

print(f"Tabla exportada por ExportMosaicDatasetPaths: {exported_mosaic_paths_table}")
print(f"Registros exportados de paths del mosaic dataset: {len(mosaic_paths_df)}")
print(f"Campos candidatos de path/name en tabla exportada: {candidate_path_fields}")
print(f"Entradas normalizadas del inventario: {len(mosaic_image_inventory_df)}")

display(mosaic_paths_fields_df)
display(mosaic_paths_df.head(25))
display(mosaic_image_inventory_df.head(25))

Tabla exportada por ExportMosaicDatasetPaths: C:\Users\ESRLRI~1\AppData\Local\Temp\scratch.gdb\mosaic_paths_20260612_162417
Registros exportados de paths del mosaic dataset: 926
Campos candidatos de path/name en tabla exportada: ['Path']
Entradas normalizadas del inventario: 914


,name,alias,type,length,required,nullable
0,OID,OID,OID,4,True,False
1,SourceOID,SourceOID,Integer,4,False,True
2,Path,Path,String,256,False,True


,OID,SourceOID,Path
0,1,1779,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...
1,2,17123,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
2,3,17124,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
3,4,17125,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
4,5,17126,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
5,6,17127,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
6,7,17128,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
7,8,17136,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
8,9,17137,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
9,10,17138,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...


,mosaic_row_index,source_field,source_value,mosaic_path,mosaic_file_name,mosaic_stem,mosaic_key,mosaic_file_key,mosaic_stem_key
0,0,Path,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,//amssclgis10.ams.gmams.cl/cl_mlp_pao/chacay_e...,cl_mlp_pao_if_ortho_26_01_07_ed2.tif,cl_mlp_pao_if_ortho_26_01_07_ed2,amssclgis10_ams_gmams_cl_cl_mlp_pao_chacay_el_...,cl_mlp_pao_if_ortho_26_01_07_ed2_tif,cl_mlp_pao_if_ortho_26_01_07_ed2
1,1,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe03_c0000d54f,ov_i9a2_l01_r0000fe03_c0000d54f,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe03_c0000d54f,ov_i9a2_l01_r0000fe03_c0000d54f
2,2,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe03_c0000d550,ov_i9a2_l01_r0000fe03_c0000d550,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe03_c0000d550,ov_i9a2_l01_r0000fe03_c0000d550
3,3,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe03_c0000d551,ov_i9a2_l01_r0000fe03_c0000d551,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe03_c0000d551,ov_i9a2_l01_r0000fe03_c0000d551
4,4,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe03_c0000d556,ov_i9a2_l01_r0000fe03_c0000d556,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe03_c0000d556,ov_i9a2_l01_r0000fe03_c0000d556
5,5,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe03_c0000d557,ov_i9a2_l01_r0000fe03_c0000d557,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe03_c0000d557,ov_i9a2_l01_r0000fe03_c0000d557
6,6,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe03_c0000d558,ov_i9a2_l01_r0000fe03_c0000d558,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe03_c0000d558,ov_i9a2_l01_r0000fe03_c0000d558
7,7,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe04_c0000d54f,ov_i9a2_l01_r0000fe04_c0000d54f,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe04_c0000d54f,ov_i9a2_l01_r0000fe04_c0000d54f
8,8,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe04_c0000d550,ov_i9a2_l01_r0000fe04_c0000d550,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe04_c0000d550,ov_i9a2_l01_r0000fe04_c0000d550
9,9,Path,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...,//amssclgis08.ams.gmams.cl/cl_mlp_pao/01_proye...,ov_i9a2_l01_r0000fe04_c0000d551,ov_i9a2_l01_r0000fe04_c0000d551,amssclgis08_ams_gmams_cl_cl_mlp_pao_01_proyect...,ov_i9a2_l01_r0000fe04_c0000d551,ov_i9a2_l01_r0000fe04_c0000d551


## 3. Comparar nombres esperados contra el mosaico

`load_status = nueva_candidata` significa que el nombre esperado por el script no aparece en el mosaico. El triage separa las candidatas donde la fecha ya existe en el mosaico, porque ahi puede haber un cambio de nomenclatura en la carga manual.


In [22]:
input_expected_names_df = add_expected_names(ortho_input_images_df)
input_vs_mosaic_df = add_mosaic_match(input_expected_names_df, mosaic_image_inventory_df)
triage_df = add_triage(input_vs_mosaic_df, mosaic_image_inventory_df)

script_new_df = triage_df[triage_df["load_status"] == "nueva_candidata"].copy()
script_new_control_df = script_new_df[script_new_df["is_in_control_folder"]].copy()
high_confidence_new_control_df = script_new_control_df[script_new_control_df["triage_status"] == "nueva_alta_confianza"].copy()
same_date_review_control_df = script_new_control_df[script_new_control_df["triage_status"] == "revisar_fecha_existente_en_mosaico"].copy()
review_or_discard_control_df = triage_df[
    triage_df["is_in_control_folder"]
    & triage_df["load_status"].isin(["sin_fecha", "sin_sector", "descartar_posible_plano"])
].copy()

print("Estados globales:")
display(triage_df["load_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "load_status"}))

print("Triage dentro del subfolder de control:")
display(triage_df[triage_df["is_in_control_folder"]]["triage_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "triage_status"}))

Estados globales:


,load_status,count
0,nueva_candidata,49
1,descartar_posible_plano,9
2,sin_fecha,7
3,ya_cargada,1


Triage dentro del subfolder de control:


,triage_status,count
0,revisar_fecha_existente_en_mosaico,14
1,descartar_posible_plano,9
2,revisar_fecha_o_nombre,6
3,nueva_alta_confianza,3
4,ya_cargada,1


## 4. Resumen ejecutivo y muestras para analisis

La tabla de resumen responde: cuantas imagenes dice el script que son nuevas y cuantas caen dentro del directorio de control del cliente. Las muestras ayudan a ver si las que no aparecen en el mosaico fueron cargadas con otro nombre.


In [23]:
summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "input_root", "value": PATH_INPUT_RAIZ},
    {"metric": "client_control_subfolder", "value": SUBFOLDER_CONTROL_CLIENTE},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "input_images_count", "value": len(input_images_df)},
    {"metric": "root_ortho_tif_count", "value": len(ortho_input_images_df)},
    {"metric": "control_ortho_tif_count", "value": len(control_ortho_images_df)},
    {"metric": "exported_mosaic_paths_table", "value": exported_mosaic_paths_table},
    {"metric": "mosaic_exported_paths_count", "value": len(mosaic_paths_df)},
    {"metric": "mosaic_inventory_count", "value": len(mosaic_image_inventory_df)},
    {"metric": "script_new_count_all_root", "value": len(script_new_df)},
    {"metric": "script_new_count_inside_control", "value": len(script_new_control_df)},
    {"metric": "high_confidence_new_inside_control", "value": len(high_confidence_new_control_df)},
    {"metric": "same_date_review_inside_control", "value": len(same_date_review_control_df)},
    {"metric": "review_or_discard_inside_control", "value": len(review_or_discard_control_df)},
]

for status, count in triage_df["load_status"].value_counts(dropna=False).items():
    summary_rows.append({"metric": f"load_status_{status}", "value": int(count)})

for status, count in triage_df[triage_df["is_in_control_folder"]]["triage_status"].value_counts(dropna=False).items():
    summary_rows.append({"metric": f"control_triage_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

sample_columns = [
    "triage_status", "load_status", "file_name", "relative_path", "expected_name",
    "expected_date_token", "expected_sector", "date_exists_in_mosaic",
    "same_date_mosaic_examples", "matched_mosaic_path"
]

print("Muestra: nuevas de alta confianza dentro del control")
display(high_confidence_new_control_df[sample_columns].head(30))

print("Muestra: dice nueva, pero la fecha ya existe en el mosaico")
display(same_date_review_control_df[sample_columns].head(30))

print("Muestra: revision o descarte dentro del control")
display(review_or_discard_control_df[sample_columns].head(30))

,metric,value
0,run_timestamp,20260612_162412
1,input_root,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...
2,client_control_subfolder,20260206_Geosupport_primera entrega
3,mosaic_dataset,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
4,input_images_count,1999
5,root_ortho_tif_count,66
6,control_ortho_tif_count,33
7,exported_mosaic_paths_table,C:\Users\ESRLRI~1\AppData\Local\Temp\scratch.g...
8,mosaic_exported_paths_count,926
9,mosaic_inventory_count,914


Muestra: nuevas de alta confianza dentro del control


,triage_status,load_status,file_name,relative_path,expected_name,expected_date_token,expected_sector,date_exists_in_mosaic,same_date_mosaic_examples,matched_mosaic_path
0,nueva_alta_confianza,nueva_candidata,GEOSP-TRN-001426_GS_ORTOFOTO EB2 28-12-2025.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_25_12_28_eb2,25_12_28,eb2,False,,None
2,nueva_alta_confianza,nueva_candidata,GEOSP-TRN-001553_GS_ORTOFOTO_EB2_20-12-2025.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_25_12_20_eb2,25_12_20,eb2,False,,None
32,nueva_alta_confianza,nueva_candidata,SIN_ID_GS_ORTOFOTO_ESTACION DE VALVULAS N°1 (p...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_05_29_estacion_de_valvu...,26_05_29,estacion_de_valvulas_n_1,False,,None


Muestra: dice nueva, pero la fecha ya existe en el mosaico


,triage_status,load_status,file_name,relative_path,expected_name,expected_date_token,expected_sector,date_exists_in_mosaic,same_date_mosaic_examples,matched_mosaic_path
3,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-001642_ORTOFOTO_EB2_03-01-2026.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_01_03_eb2,26_01_03,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/el_mauro...,None
4,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-001732_GS_ORTOFOTO_EB2_17-01-26.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_01_17_eb2,26_01_17,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/el_mauro...,None
6,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-001868_GS_ORTOFOTO EB2 04-02-2026.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_02_04_eb2,26_02_04,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/chacay_e...,None
8,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-002016_GS_ORTOFOTO_EB2_26-02-26.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_02_26_eb2,26_02_26,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/el_mauro...,None
9,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-002096_GS_ORTOFOTO_EB2_07-03-2026.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_03_07_eb2,26_03_07,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/el_mauro...,None
10,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-002140_GS_ORTOFOTO_EB2_13-03-26.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_03_13_eb2,26_03_13,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/chacay_d...,None
12,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-002235_GS_ORTOFOTO_EB2_27-03-26.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_03_27_eb2,26_03_27,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/chacay_d...,None
14,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-002356_GS_ORTOFOTO_EB2_12-04-26.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_04_12_eb2,26_04_12,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/el_mauro...,None
15,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-002415_GS_ORTOFOTO_EB2_190426.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_04_19_eb2,26_04_19,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/el_mauro...,None
16,revisar_fecha_existente_en_mosaico,nueva_candidata,GEOSP-TRN-002456_ORTOFOTO_CORTADA_EB2_260426.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,CL_MLP_PAO_IF_Ortho_26_04_26_eb2,26_04_26,eb2,True,//amssclgis10.ams.gmams.cl/cl_mlp_pao/puerto_p...,None


Muestra: revision o descarte dentro del control


,triage_status,load_status,file_name,relative_path,expected_name,expected_date_token,expected_sector,date_exists_in_mosaic,same_date_mosaic_examples,matched_mosaic_path
1,revisar_fecha_o_nombre,sin_fecha,GEOSP-TRN-001492_GS_ORTOFOTO_Estacion de bombe...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
5,revisar_fecha_o_nombre,sin_fecha,GEOSP-TRN-001784_GS_ORTOFOTO_Estacion de bombe...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
7,revisar_fecha_o_nombre,sin_fecha,GEOSP-TRN-001975_GS_ORTOFOTO_Estacion de bombe...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
11,revisar_fecha_o_nombre,sin_fecha,GEOSP-TRN-002200_ORTOFOTO COMPLETA.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
13,revisar_fecha_o_nombre,sin_fecha,GEOSP-TRN-002300_ortofoto completa.tif,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
17,descartar_posible_plano,descartar_posible_plano,GEOSP-TRN-002504_1001-03-T-CS-202-4310-C-DW-23...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
19,descartar_posible_plano,descartar_posible_plano,GEOSP-TRN-002517_1001-03-T-CS-202-5600-C-DW-11...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
21,descartar_posible_plano,descartar_posible_plano,GEOSP-TRN-002551_1001-03-T-CS-202-5600-C-DW-11...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
22,descartar_posible_plano,descartar_posible_plano,GEOSP-TRN-002562_1001-03-T-CS-202-5600-C-DW-11...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None
23,descartar_posible_plano,descartar_posible_plano,GEOSP-TRN-002566_1001-03-T-CS-202-5600-C-DW-11...,20260206_Geosupport_primera entrega\SOLO_TIF_0...,None,None,None,False,,None


## 5. Exportar resultados

Se exportan CSV y SQLite para analizarlos fuera del servidor.


In [24]:
dataframes_to_export = {
    "summary": summary_df,
    "input_images": input_images_df,
    "ortho_input_images": ortho_input_images_df,
    "control_ortho_images": control_ortho_images_df,
    "mosaic_paths_fields": mosaic_paths_fields_df,
    "mosaic_paths_export": mosaic_paths_df,
    "mosaic_image_inventory": mosaic_image_inventory_df,
    "input_vs_mosaic": input_vs_mosaic_df,
    "triage": triage_df,
    "script_new_all_root": script_new_df,
    "script_new_inside_control": script_new_control_df,
    "high_confidence_new_inside_control": high_confidence_new_control_df,
    "same_date_review_inside_control": same_date_review_control_df,
    "review_or_discard_inside_control": review_or_discard_control_df,
}

exported_results = export_results(dataframes_to_export, output_results_dir, run_timestamp)
exported_results_df = pd.DataFrame([{"name": name, "path": str(path)} for name, path in exported_results.items()])

print(f"Resultados exportados en: {output_results_dir}")
display(exported_results_df)

Resultados exportados en: c:\Users\esrlrivero_adm\Documents\amsa-pao-geosupport\outputs\auditoria_mosaico\20260612_162412


,name,path
0,summary_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
1,input_images_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
2,ortho_input_images_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
3,control_ortho_images_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
4,mosaic_paths_fields_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
5,mosaic_paths_export_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
6,mosaic_image_inventory_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
7,input_vs_mosaic_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
8,triage_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
9,script_new_all_root_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...


In [25]:
##leo
